In [ ]:
import pandas as pd 
import numpy as np 
import plotly.express as px 
import plotly.graph_objects as go 
import os

In [ ]:
data_path = os.getenv('DATA_PATH')
reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv', low_memory=False)

# EDA

In [ ]:
reviews[reviews['description'].notna()].shape

In [ ]:
reviews.iloc[0]

In [ ]:
reviews.info()

In [ ]:
cols = reviews.columns.tolist()
miss_values = pd.Series([int(reviews[col].isna().sum()) for col in cols])

def outliers_number(data: pd.Series, iqr=True,  modified=False, threshold=3, threshold_modified=4.5):
    if iqr:
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        IQR = q3-q1

        return data[(data > q3+IQR*1.5) | (data < q1-IQR*1.5)].index
    else:
        if modified:
            median = data.median()
            mad = np.median(np.abs(data - median))
            
            modified_z_scores = 0.6745 * (data - median) / mad

            return data[np.abs(modified_z_scores) > threshold_modified].index
        else:
            z_score = (data - data.mean()) / data.std()
            return data[np.abs(data) > threshold].index
        
def outlier_len(iqr, modified):
    outlier_iqr = []
    for col in cols:
        if (type(reviews[col][0]) in [np.int64, np.float64]):
            outlier_iqr.append(len(outliers_number(reviews[col], iqr, modified)))
        else:
            outlier_iqr.append(np.nan)

    return pd.Series(outlier_iqr)

pd.DataFrame(index=reviews.columns.tolist(),
             data={
            'miss value': miss_values.values,
            "type": reviews.dtypes.values,
            'outlier_iqr': outlier_len(True, False).values,
            'outlier_Z-score': outlier_len(False, False).values,
            'outlier_Z-score modified': outlier_len(False, True).values,
            'Most repeated': [reviews[col].value_counts().index[0] for col in cols]
            })

In [ ]:
reviews[(reviews['productId']== 1715746) & (reviews['description'].notna())][['description', 'star', 'updatedAt']
                                ].sort_values(by='updatedAt', ascending=False) # type: ignore

In [ ]:
reviews_count_per_product = reviews.groupby('productId')['_id'].agg(['count']).reset_index().sort_values(
    by='count', ascending=False)

In [ ]:
fig = px.bar(reviews_count_per_product, x='productId', y='count', text='count')

fig.show()

In [ ]:
sample = reviews[reviews['description'].notna()].sample(10000).copy()

In [ ]:
drop_cols = [
    'id', 
    'user_id_of_user', 'hashId', 'hash_id_of_user', 'reason_ids[0]',
    'reason_ids[1]', 'reason_ids[2]', 'reason_ids[3]', 'reason_ids[4]',
    'reason_ids[5]', 'reason_ids[6]', 'reason_ids[7]', 'variation_metadata'
]

In [ ]:
data = reviews.drop(columns=drop_cols).sample(100000)

In [ ]:
star_by_time = data[['star', 'createdAt']].copy()
star_by_time.loc[:, 'createdAt'] = pd.to_datetime(star_by_time['createdAt'])
star_by_time.loc[:, 'star'] = star_by_time['star'].map({
    1:'1',
    2:'1',
    3:'1',
    4:'2',
    5:'3'
})

In [ ]:
star_by_time = star_by_time.sort_values(by='createdAt', ascending=False)
star_by_time['season'] = pd.PeriodIndex(star_by_time['createdAt'],freq='Q')

In [ ]:
star_by_season = star_by_time.groupby('season')['star'].value_counts(normalize=True).mul(100).reset_index(name='percentage')
star_by_season = star_by_season.sort_values(by=['season', 'star'], ascending=[True, True])

In [ ]:
star_by_season['season'] = star_by_season['season'].map(lambda x: str(f"{x.year}-season{x.quarter}"))

In [ ]:
fig = px.bar(star_by_season, 
             x='season', 
             y='percentage', 
             color='star', 
             text='percentage')

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')

fig.show()

# time series for number of comments in day

In [ ]:
reviews['updatedAt'].isna().sum()

In [ ]:
reviews['updatedAt'] = pd.to_datetime(reviews['updatedAt'])
reviews['updatedAt_day'] = reviews['updatedAt'].dt.date
df = reviews.groupby('updatedAt_day').size().reset_index(name='count')
df['updatedAt_day'] = pd.to_datetime(df['updatedAt_day'])
df['day-of-week'] = df['updatedAt_day'].dt.day_name()
df = df.iloc[:-30] 

In [ ]:
fig = px.line(df, x='updatedAt_day' , y='count', hover_data=['day-of-week'])
fig.show()

## decomposition

In [ ]:
df['Rolling_Mean'] = df['count'].rolling(window=7).mean()
df['Rolling_Std'] = df['count'].rolling(window=7).std()

fig = go.Figure()

fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=df['count'], mode='lines', name='Original count', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=df['Rolling_Mean'], mode='lines', name='Rolling Mean (Trend)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=df['Rolling_Std'], mode='lines', name='Rolling Std Dev', line=dict(color='green')))


fig.update_layout(
    title="Trend and Variance Over Time",
    xaxis_title="Date",
    yaxis_title="Value",
    legend=dict(x=0, y=1),
    height=500,
    width=900
)
fig.show()

### additive

In [ ]:
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df['count'], model='additive', period=3)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=["Original", "Trend", "Seasonality", "Residuals"])

fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=df['count'], mode='lines', name="Original", line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=decomposition.trend, mode='lines', name="Trend", line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=decomposition.seasonal, mode='lines', name="Seasonality", line=dict(color='green')), row=3, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=decomposition.resid, mode='lines', name="Residuals", line=dict(color='black')), row=4, col=1)

fig.update_layout(title="Seasonal Decomposition of count", height=800, width=900, showlegend=False)

fig.show()

### multiplicative

In [ ]:
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df['count'], model='multiplicative', period=3)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=["Original", "Trend", "Seasonality", "Residuals"])

fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=df['count'], mode='lines', name="Original", line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=decomposition.trend, mode='lines', name="Trend", line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=decomposition.seasonal, mode='lines', name="Seasonality", line=dict(color='green')), row=3, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=decomposition.resid, mode='lines', name="Residuals", line=dict(color='black')), row=4, col=1)

fig.update_layout(title="Seasonal Decomposition of count", height=800, width=900, showlegend=False)

fig.show()


### STL

In [ ]:
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL

stl = STL(df['count'], period=3) 
decomposition = stl.fit()

trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=["Original", "Trend", "Seasonality", "Residuals"])

fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=df['count'], mode='lines', name="Original", line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=trend, mode='lines', name="Trend", line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=seasonal, mode='lines', name="Seasonality", line=dict(color='green')), row=3, col=1)
fig.add_trace(go.Scatter(x=df['updatedAt_day'], y=residual, mode='lines', name="Residuals", line=dict(color='black')), row=4, col=1)

fig.update_layout(title="STL Decomposition of count", height=800, width=900, showlegend=False)

fig.show()

### Prophet

In [ ]:
from prophet import Prophet
from plotly.subplots import make_subplots

df_prophet = df.reset_index()[['updatedAt_day', 'count']].rename(columns={'updatedAt_day': 'ds', 'count': 'y'})

model = Prophet()
model.fit(df_prophet)

forecast = model.predict(df_prophet)

df_prophet['Trend'] = forecast['trend']
df_prophet['Seasonality'] = forecast['additive_terms']
df_prophet['Residuals'] = df_prophet['y'] - (df_prophet['Trend'] + df_prophet['Seasonality'])

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=["Original", "Trend", "Seasonality", "Residuals"])

fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'], mode='lines', name="Original", line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['Trend'], mode='lines', name="Trend", line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['Seasonality'], mode='lines', name="Seasonality", line=dict(color='green')), row=3, col=1)
fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['Residuals'], mode='lines', name="Residuals", line=dict(color='black')), row=4, col=1)

fig.update_layout(title="Prophet Decomposition of count", height=800, width=1000, showlegend=False)

fig.show()


## predict

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt 

train = df.iloc[:-100]
test = df.iloc[-100:]

test_index = test.index


### lstm

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['count']])

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

seq_length = 100
X, y = create_sequences(scaled_data, seq_length)

X_train, X_test = X[:-100], X[-100:]
y_train, y_test = y[:-100], y[-100:]

model = Sequential([
    LSTM(50, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
    LSTM(50, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.fit(X_train, y_train, epochs=20, batch_size=8, validation_data=(X_test, y_test), verbose=1)

y_pred = model.predict(X_test)

y_pred = scaler.inverse_transform(y_pred)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

mae_lstm = mean_absolute_error(y_test, y_pred)
print(f"LSTM MAE: {mae_lstm:.2f}")

In [ ]:
y_pred = [x[0] for x in y_pred]
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train['count'], mode='lines', name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test.index, y=test['count'], mode='lines', name='Test (Actual)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=test.index, y=y_pred, mode='lines', name='LSTM Forecast', line=dict(color='red', dash='dot')))

fig.update_layout(title="Time Series Forecasting with LSTM", xaxis_title="Date", yaxis_title="count", height=600, width=1000, showlegend=True)
fig.show()

In [ ]:
scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(train[['count']])
scaled_test = scaler.transform(test[['count']])

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 3
X_train, y_train = create_sequences(scaled_train, seq_length)
X_test, y_test = create_sequences(scaled_test, seq_length)

model_lstm = Sequential([
    LSTM(50, activation='relu',  return_sequences=True, input_shape=(seq_length, 1)),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mse')

model_lstm.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test, y_test), verbose=0)

lstm_pred_scaled = model_lstm.predict(X_test)
lstm_forecast = scaler.inverse_transform(lstm_pred_scaled)

# lstm_forecast = [x[0] for x in lstm_forecast]

fig = go.Figure()

fig.add_trace(go.Scatter(x=train.index, y=train['count'], mode='lines', name='Train Data'))
fig.add_trace(go.Scatter(x=test.index, y=test['count'], mode='lines', name='Actual (Test Data)', line=dict(color='black')))
fig.add_trace(go.Scatter(x=test.index, y=lstm_forecast, mode='lines', name='LSTM Prediction', 
                         line=dict(color='orange', dash='dash')))

fig.update_layout(
    title="LSTM: Prediction vs. Actual",
    xaxis_title='Date',
    yaxis_title='Count',
    legend_title='Legend',
)

fig.show()


### sarima

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

train = df.iloc[:-100]
test = df.iloc[-100:]
model = SARIMAX(train['count'], order=(0, 0, 2), seasonal_order=(0, 1, 0, 100))
model_fit = model.fit()

forecast = model_fit.forecast(steps=100)
test['Forecast'] = forecast.values

mae_sarima = mean_absolute_error(test['count'], test['Forecast'])
print(f"SARIMA MAE: {mae_sarima:.2f}")

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['updatedAt_day'], y=train['count'], mode='lines', name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test['updatedAt_day'], y=test['count'], mode='lines', name='Test (Actual)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=test['updatedAt_day'], y=test['Forecast'], mode='lines', name='SARIMA Forecast', line=dict(color='red', dash='dot')))

fig.update_layout(title="Time Series Forecasting with SARIMA", xaxis_title="Date", yaxis_title="count", height=600, width=1000, showlegend=True)
fig.show()

### XGBoost

In [ ]:
# Create time-based features for XGBoost
train['Month'] = train['updatedAt_day'].dt.month
train['Year'] = train['updatedAt_day'].dt.year
train['day'] = train['updatedAt_day'].dt.day_of_week
test['Month'] = test['updatedAt_day'].dt.month
test['Year'] = test['updatedAt_day'].dt.year
test['day'] = test['updatedAt_day'].dt.day_of_week

X_train_xgb = train[['Month', 'Year', 'day']]
y_train_xgb = train['count']
X_test_xgb = test[['Month', 'Year', 'day']]

model_xgb = XGBRegressor(n_estimators=1001, learning_rate=0.9)
model_xgb.fit(X_train_xgb, y_train_xgb)

xgb_forecast = model_xgb.predict(X_test_xgb)

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['updatedAt_day'], y=train['count'], mode='lines', name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test['updatedAt_day'], y=test['count'], mode='lines', name='Test (Actual)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=test['updatedAt_day'], y=xgb_forecast, mode='lines', name='SARIMA Forecast', line=dict(color='red', dash='dot')))

fig.update_layout(title="Time Series Forecasting with SARIMA", xaxis_title="Date", yaxis_title="count", height=600, width=1000, showlegend=True)
fig.show()

### lightgbm

In [ ]:
import lightgbm as lgb

df['Month'] = df['updatedAt_day'].dt.month
df['Weekday'] = df['updatedAt_day'].dt.weekday
df['Year'] = df['updatedAt_day'].dt.year
# df['Lag_1'] = df['count'].shift(1)
# df.dropna(inplace=True)  # Remove NaN values caused by shifting

X = df[['Month', 'Weekday', 'Year']]
y = df['count']

X_train, X_test = X.iloc[:-100], X.iloc[-100:]
y_train, y_test = y.iloc[:-100], y.iloc[-100:]

model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae_lgbm = mean_absolute_error(y_test, y_pred)
print(f"LightGBM MAE: {mae_lgbm:.2f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['updatedAt_day'], y=train['count'], mode='lines', name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test['updatedAt_day'], y=y_test, mode='lines', name='Test (Actual)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=test['updatedAt_day'], y=y_pred, mode='lines', name='LightGBM Forecast', line=dict(color='orange', dash='dot')))

fig.update_layout(title="Time Series Forecasting with LightGBM", xaxis_title="Date", yaxis_title="count", height=600, width=1000, showlegend=True)
fig.show()

# time series for number of comment in hour

In [ ]:
reviews['updatedAt'].isna().sum()

In [ ]:
reviews['updatedAt'] = pd.to_datetime(reviews['updatedAt'])
reviews['updatedAt_hour'] = reviews['updatedAt'].dt.hour

In [ ]:
df_hour = reviews.groupby('updatedAt_hour').size().reset_index(name='count')
df_hour.head()

In [ ]:
fig = px.line(df_hour, x='updatedAt_hour', y='count')
fig.show()